# R7.1

In [2]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-02.parquet"
df = pd.read_parquet(url)

In [3]:
from pipeline import build_pipeline

model_pipeline = build_pipeline()

In [5]:
from sklearn.model_selection import train_test_split
X = df.drop("total_amount", axis=1)
y = df["total_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model_pipeline.fit(X_train, y_train)


print("Model trained successfully")

Model trained successfully


In [6]:
#Wrong version
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

X_leak = X.copy()

num_cols = [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "extra",
    "Airport_fee",
    "congestion_surcharge",
    "cbd_congestion_fee"
]

# WRONG: fit on entire dataset
imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()

X_leak[num_cols] = imputer.fit_transform(X_leak[num_cols])
X_leak[num_cols] = scaler.fit_transform(X_leak[num_cols])

leak_model = LinearRegression()

leak_score = cross_val_score(
    leak_model,
    X_leak[num_cols],
    y,
    cv=5,
    scoring="r2"
)

print("Leakage Score:", leak_score.mean())

Leakage Score: 0.9818566813989896


In [8]:
#right one
from sklearn.model_selection import cross_val_score

pipeline_score = cross_val_score(
    model_pipeline,
    X,
    y,
    cv=3,
    scoring="r2"
)

print("Pipeline Score:", pipeline_score.mean())

Pipeline Score: 0.9752858468901517


In [9]:
print("Leakage Score :", leak_score.mean())
print("Pipeline Score:", pipeline_score.mean())
print("Difference    :", leak_score.mean() - pipeline_score.mean())

Leakage Score : 0.9818566813989896
Pipeline Score: 0.9752858468901517
Difference    : 0.006570834508837886


# R7.2 Unseen Category Test

In [10]:

model_pipeline.fit(X, y)

test_row = pd.DataFrame({
    "trip_distance":[5],
    "fare_amount":[20],
    "tip_amount":[2],
    "tolls_amount":[0],
    "extra":[0],
    "Airport_fee":[0],
    "congestion_surcharge":[2.5],
    "cbd_congestion_fee":[0.75],
    "VendorID":[999],
    "payment_type":[999],
    "RatecodeID":[999],
    "store_and_fwd_flag":["Z"],
    "PULocationID":[999],
    "DOLocationID":[999]
})

pred = model_pipeline.predict(test_row)

print(pred)

[30.96847224]


In [13]:
num_cols = [
    "passenger_count",
    "congestion_surcharge",
    "Airport_fee"
]


cat_cols = [
    "RatecodeID",
    "store_and_fwd_flag"
]

# R9

In [14]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LinearRegression
import numpy as np

for strategy in ["mean", "median"]:

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy=strategy)),
        ("scaler", RobustScaler())
    ])

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        cv=3,      # 5 ki jagah 3 rakho, faster chalega
        scoring="r2",
        n_jobs=-1
    )

    print(strategy)
    print("Mean R2 =", np.mean(scores))
    print()

mean
Mean R2 = 0.32217755685701316

median
Mean R2 = 0.2932161687362711

